Priprema kombinovani dataset (EgoHands + filtered), ucitava pocetne tezine i radi brzi fine-tuning segmentacionog modela uz evaluaciju.

<!-- # 03 — combined_seg_fast (brzi combined fine-tune)

| Metrika | Val (ep. 10) | Test (EgoHands) |
|---------|--------------|-----------------|
| Mask mAP50 | **12.4%** | **90.0%** |
| Mask mAP50-95 | **6.4%** | **56.1%** |

**Inicijalne težine:** `combined_seg/weights/last.pt` (fallback: `cpu_quick_seg/best.pt`)  
**Run folder:** `runs/segment/.../combined_seg_fast/`

**Prerequisites:** `00`, `merge_datasets`, težine iz `02` ili `01` -->


In [28]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
SRC_DIR = NOTEBOOK_DIR
os.chdir(SRC_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ultralytics import YOLO

EGOHANDS_DIR = SRC_DIR / "data" / "egohands_yolo"
EGOHANDS_YAML = EGOHANDS_DIR / "data.yaml"
RUNS_DIR = SRC_DIR / "runs" / "segment" / "instance_segmentation" / "runs" / "segment"
DEVICE = "cpu"


In [29]:
train_dir = EGOHANDS_DIR / "images" / "train"
if not train_dir.exists() or not any(train_dir.iterdir()):
    raise FileNotFoundError(
        "EgoHands dataset nije spreman. Prvo pokreni 00_data_processing.ipynb"
    )

import importlib
import merge_datasets
importlib.reload(merge_datasets)
from merge_datasets import prepare_combined_dataset

FILTERED_DIR = SRC_DIR / "data" / "yolo_dataset_filtered"
COMBINED_YAML = prepare_combined_dataset(
    egohands_dir=EGOHANDS_DIR,
    filtered_dir=FILTERED_DIR,
    output_yaml=SRC_DIR / "data" / "combined_data.yaml",
)

resume_weights = RUNS_DIR / "combined_seg" / "weights" / "last.pt"
if not resume_weights.exists():
    resume_weights = RUNS_DIR / "cpu_quick_seg" / "weights" / "best.pt"
if not resume_weights.exists():
    raise FileNotFoundError(
        "Nedostaju tezine combined_seg ili cpu_quick_seg. "
        "Pokreni 01 i/ili 02 pre ovog notebooka."
    )
print(f"Combined YAML: {COMBINED_YAML}")
print(f"Resume weights: {resume_weights}")


Combined dataset config: C:\ml-fingers-matf\src\yolo_instance_segmentation\data\combined_data.yaml
  EgoHands train/val:   3300 / 700
  Filtered train/val:   7415 / 3125
  Total train/val:      10715 / 3825
  Test (EgoHands only): 800
Combined YAML: C:\ml-fingers-matf\src\yolo_instance_segmentation\data\combined_data.yaml
Resume weights: C:\ml-fingers-matf\src\yolo_instance_segmentation\runs\segment\instance_segmentation\runs\segment\cpu_quick_seg\weights\best.pt


In [30]:
RUN_NAME = "combined_seg_fast"

model = YOLO(str(resume_weights.resolve()))
model.train(
    task="segment",
    data=str(COMBINED_YAML),
    epochs=10,
    imgsz=416,
    batch=4,
    device=DEVICE,
    workers=0,
    cache=True,
    amp=False,
    freeze=10,
    fraction=0.25,
    val=False,
    plots=False,
    project="instance_segmentation/runs/segment",
    name=RUN_NAME,
    exist_ok=True,
)


New https://pypi.org/project/ultralytics/8.4.90 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.56  Python-3.11.15 torch-2.12.0+cpu CPU (AMD Ryzen 5 7535HS with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\ml-fingers-matf\src\yolo_instance_segmentation\data\combined_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.25, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\ml-fin

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002AA0C166150>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.0410

In [ ]:
from evaluate_yolo import evaluate_model_quality, find_best_weights

RUN_NAME = "combined_seg_fast"
metrics = evaluate_model_quality(
    weights=find_best_weights(RUN_NAME),
    data_yaml=COMBINED_YAML,
    split="test",
    device=DEVICE,
    show_samples=6,
)
metrics


In [ ]:
from evaluate_yolo import find_best_weights, predict_custom_images

MY_TEST = (EGOHANDS_DIR / "myTest").resolve()
if MY_TEST.exists():
    predict_custom_images(
        weights=find_best_weights("combined_seg_fast"),
        source=MY_TEST,
        device=DEVICE,
        conf=0.15,
        save_dir=MY_TEST / "predictions_combined",
        show=True,
    )
else:
    print(f"myTest folder ne postoji: {MY_TEST}")
